## Vanilla MobileNetV3 (Small and Large) Implementation

**AIM: Build and train an image classifier to detect images from different animal species using a Custom MobileNetV3 (Small and Large) Model in TensorFlow.**

### Objectives

- Data visualisation
- Data preprocessing and image augmentation
- Replicate the MobileNetV3 (Small and Large) architecture for model development.
- Compile and train the model
- Add early stopping callback
- Save and load the model
- Model evaluation.
- Make predictions on new data using the trained model.

### Pre-requisite
- Google collaboratry or Jupyter Notebook
- animal-image-classification-dataset
- TensorFlow2

In [ ]:
# Import basic libraries
import os
import sys
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

In [ ]:
# Set seed for reproducibility

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
gpus= tf.config.list_physical_devices()

In [ ]:

gpus

In [ ]:
logical_devices = tf.config.list_logical_devices()
logical_devices

In [ ]:
# Check tenorflow version
print("TensorFlow Version", tf.__version__)

In [ ]:
## Set the base path
base_dir = "../../datasets/dog_vs_cats"
base_dir = pathlib.Path(base_dir)
base_dir

In [ ]:
# Train directory
train_dir = base_dir / "train"
train_dir

In [ ]:
# Validation directory
test_dir = base_dir / "test"
test_dir

In [ ]:
# Validation directory
validation_dir = base_dir / "validation"
validation_dir

In [ ]:
## Set Hyperparameters

IMAGE_HEIGHT, IMAGE_WIDTH = 128, 128
BATCH_SIZE = 32
EPOCHS = 300

In [ ]:
# Load the training dataset

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

In [ ]:
# Load the validation dataset

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
)

In [ ]:
# Get the class names
class_names = train_dataset.class_names
class_names

In [ ]:
# Get the total number of classes
num_classes = len(class_names)
num_classes

In [ ]:
# Sanity check

for images, labels in train_dataset.take(1):
    fixed_images = images.numpy()
    fixed_labels = labels.numpy()


# Visualisations
# No matter how many times you run this cell, the images won change because of teh above

plt.figure(figsize=(12, 12))
for i in range(16):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(fixed_images[i].astype("uint8"))
    plt.title(class_names[fixed_labels[i]])
    plt.axis("off")

In [ ]:
# Performance optimization

### Vanilla MobileNetV3 Implementation

In [ ]:
INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH) + (3, )
INPUT_SHAPE

### Utility Scripts

In [2]:
# TensorFlow related imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### Ativations: Hard Sigmoid and Hard Swish

In [3]:
# Hard Swish Activation
class HardSwish(layers.Layer):
    """
    Hard Swish activation function: x * ReLU(x + 3) / 6
    More efficient approximation of Swish for mobile devices
    """

    def __init__(self, **kwargs):
        super(HardSwish, self).__init__(**kwargs)

    def call(self, inputs):
        return inputs * tf.nn.relu6(inputs + 3.0) / 6.0
    
    def get_config(self):
        return super(HardSwish, self).get_onfig()

In [4]:
# Hard Sigmoid Activation
class HardSigmoid(layers.Layer):
    """
    Hard Sigmoid activation function: ReLU6(x + 3) / 3
    More efficient aproximation of Sigmoid for mobile devices.
    """
    def __init__(self, **kwargs):
        super(HardSigmoid, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.nn.relu6(inputs + 3.0) / 6.0
    
    def get_config(self):
        return super(HardSigmoid, self).get_config()

### Squeeze-and-Excitation (SE) Block

In [5]:
class SqueezeExcitation(layers.Layer):
    """
    Squeeze-and-Excitation block for channel attention

    Args:
        filters: Number of filters in the input
        se_ratio: Squeeze ratio for dimensionality redction (default: 0.25)
    """

    def __init__(self, filters, se_ratio=0.25, **kwargs):
        super(SqueezeExcitation, self).__init__(**kwargs)
        self.filters = filters
        self.se_ratio = se_ratio
        self.reduced_filters = max(1, int(filters * se_ratio))

    def build(self, input_shape):
        # Global Average Pooling
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)

        # FC Layers
        self.fc1 = layers.Conv2D(self.reduced_filters,
                                 kernel_size=1,
                                 padding="same",
                                 use_bias=True,
                                 name="se_Reduced")
        
        self.fc2 = layers.Conv2D(self.filters,
                                 kernel_size=1,
                                 padding="same",
                                 use_bias=True,
                                 name="se_expand")
        
        self.relu = layers.ReLU()
        self.hard_sigmoid = HardSigmoid()

    def call(self, inputs):
        # Squeeze: Global pooling
        x = self.gap(inputs)

        # Excitation: FC -> ReLU -> FC -> Hard-Sigmoid
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.hard_sigmoid(x)

        # Sale
        return inputs * x
    
    def get_config(self):
        config = super(SqueezeExcitation, self).get_config()
        config.update({
            "filters": self.filters,
            "se_ratio": self.se_ratio
        })

        return config

### Inverted Residual Block (MBConv)

In [6]:
class InvertedResidualBlock(layers.Layer):
    """
    Inverted Residual Blok with expansion, depthwise convolution, and projetion.
    This is the core building block of MobileNetV3

    Args:
        expansion: Expansion factor for the intermediate channels
        filters: Number of output filters
        kernel_size: Kernel size for depthwise convolution
        stride: Stride for depthsie convolution
        se_ratio: Squeeze-Excitation ration (None to disable SE)
        ativation: Activation funtion (`RE` for ReLU, `HS` for Hard-Swish)
    """

    def __init__(self, 
                 expansion, 
                 filters, 
                 kernel_size, 
                 stride, 
                 se_ratio=None, 
                 activation="RE", 
                 **kwargs):
        super(InvertedResidualBlock, self).__init__(**kwargs)
        self.expansion = expansion
        self.filters = filters
        self.kernel_size = kernel_size
        self.stride = stride
        self.se_ratio = se_ratio
        self.activation = activation
        self.use_residual = (stride == 1)

    def build(self, input_shape):
        input_channels = input_shape[-1]
        expanded_channels = input_channels * self.expansion

        # Choose activation funtion
        if self.activation == "RE":
            act_layer = layers.ReLU()
        elif self.activation == "HS":
            act_layer = HardSwish()
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
        
        # Build layers
        self.block_layers = []

        # Expansion phase (only if expansion > 1)
        if self.expansion != 1:
            self.block_layers.extend([
                layers.Conv2D(expanded_channels, 
                              kerne_size=1, 
                              padding="same", 
                              use_bias=False,
                              name="expand_conv"
                ),
                layers.BatchNormalization(name="expand_bn"),
                act_layer

            ])
        
        # Depthwise Convolution
        self.block_layers.extend([
            layers.DepthwiseConv2D(
                kernel_size=self.kernel_size,
                strides=self.stride,
                padding="same",
                use_bias=False,
                name="depthwise_conv"
            ),
            layers.BatchNormalization(name="depthwise_bn"),
            act_layer
        ])

        # Squeeze-and-Excitation
        if self.se_ratio is not None:
            self.se_block = SqueezeExcitation(
                expanded_channels, se_ratio=self.se_ratio, name="se"
            )
        
        # Projection Phase
        self.project_conv = layers.Conv2D(
            self.filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name="project_conv"
        )
        self.project_bn = layers.BatchNormalization(name="project_bn")

        # Determine if we can use residual connection
        self.use_residual = (
            self.stride ==  1 and input_channels == self.filters
        )

    def call(self, inputs, training=None):
        x = inputs

        # Apply expandion and depthwise layers
        for layer in self.block_layers:
            x = layer(x, training=training) if isinstance(layer, layers.BatchNormalization) else layer(x)

        # Apply SE if present
        if self.se_ratio is not None:
            x = self.se_block(x, training=training)

        # Projection
        x = self.project_conv(x)
        x = self.project_bn(x, training=training)

        # Residual Connection
        if self.use_residual:
            x = layers.Add()([inputs, x])

        return x

    def get_config(self):
        config = super(InvertedResidualBlock, self).get_config()
        config.update({
            "expansion": self.expansion,
            "filters": self.filters,
            "kernel_size": self.kernel_size,
            "stride": self.stride,
            "se_ratio": self.se_ratio,
            "activation": self.activation,
        })

        return config

### MobileNetV3 Architecture

In [ ]:
def _make_divisible(v, divisor=8, min_value=None):
    """
    Ensure that all layers have a channel number dvisible by the divisor
    """

    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)

    # Make sure that round down dows not go down more thatn 10%
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

In [9]:
def MobileNetV3(
        architecture="large",
        input_shape=(224, 224, 3),
        num_classes=1000,
        width_multiplier=1.0,
        include_top=True,
        pooling=None,
        dropout_rate=0.2
):
    
    """
    Build MobileNetV3 model.

    Args:
        architecture: `large` or `small`
        input_shape: Shape of input images
        num_classes: Number of classification classes
        width_multiplier: Width multiplier for channels
        inlude_top: Whether to inlude classification head
        pooling: Pooling mode for feature extraction (`avg` or `max`)
        dropout_rate: Dropout rate before final classification layer

    Returns:
        A Keras Model Instance
    """

    if architecture not in ["large", "small"]:
        raise ValueError("Architecture must be `large` or `small`")
    
    # Define architecture configurations
    # Format: [expansion, filters, kernel_size, stride, se_ratio, activation]
    if architecture == "large":
        config = [
            # expansion, filters, kernel, stride, SE, activation
            [1,  16,  3, 1, None,  "RE"], # 112x112
            [4,  24,  3, 2, None,  "RE"], # 56x56
            [3,  24,  3, 1, None,  "RE"],
            [3,  40,  5, 2, 0.25,  "RE"], #28x28
            [3,  40,  5, 1, 0.25,  "RE"],
            [3,  40,  5, 1, 0.25,  "RE"],
            [6,  80,  3, 2, None,  "HS"], # 14x14
            [2.5, 80, 3, 1, None,  "HS"], 
            [2.3, 80, 3, 1, None,  "HS"],
            [2.3, 80, 3, 1, None,  'HS'],
            [6, 112,  3, 1, 0.25,  'HS'],
            [6, 112,  3, 1, 0.25,  'HS'],
            [6, 160,  5, 2, 0.25,  'HS'],  # 7x7
            [6, 160,  5, 1, 0.25,  'HS'],
            [6, 160,  5, 1, 0.25,  'HS'],
        ]

        last_conv_filters = 960
        last_point_filters = 1280

    else:  # Small
        config = [
            # expansion, filters, kernel, stride, SE, activation
            [1,  16,  3, 2, 0.25,  'RE'],  # 56x56
            [4.5, 24, 3, 2, None,  'RE'],  # 28x28
            [3.67, 24, 3, 1, None, 'RE'],
            [4,  40,  5, 2, 0.25,  'HS'],  # 14x14
            [6,  40,  5, 1, 0.25,  'HS'],
            [6,  40,  5, 1, 0.25,  'HS'],
            [3,  48,  5, 1, 0.25,  'HS'],
            [3,  48,  5, 1, 0.25,  'HS'],
            [6,  96,  5, 2, 0.25,  'HS'],  # 7x7
            [6,  96,  5, 1, 0.25,  'HS'],
            [6,  96,  5, 1, 0.25,  'HS'],
        ]
        last_conv_filters = 576
        last_point_filters = 1024

    
    # Input layer
    inputs = layers.Input(shape=input_shape)


    # First convolution layer
    first_conv_filters = _make_divisible(16 * width_multiplier)

    x = layers.Conv2D(
        first_conv_filters,
        kernel_size=3,
        strides=2,
        padding="same",
        use_bias=False,
        name="conv_stem"
    )(inputs)
    x = layers.BatchNormalization(name="bn_stem")(x)
    x = HardSwish(name="activation_stem")(x)

    # Build inverted residual block
    for i, (exp, filters, kernel, stride, se_ratio, activation) in enumerate(config):
        filters = _make_divisible(filters * width_multiplier)
        x = InvertedResidualBlock(
            expansion=exp,
            filters=filters,
            kernel_size=kernel,
            stride=stride,
            se_ratio=se_ratio,
            activation=activation,
            name=f"block_{i}"
        )(x)

    # Last convolution layers
    last_conv_filters = _make_divisible(last_conv_filters * width_multiplier)
    x = layers.Conv2D(
        last_conv_filters,
        kernel_size=1,
        padding="same",
        use_bias=False,
        name="conv_head",
    )(x)
    x = layers.BatchNormalization(name="bn_head")(x)
    x = HardSwish(name="activation_head")(x)

    if include_top:
        # Global Average Pooling
        x = layers.GlobalAveragePooling2D(name="avg_pool")(x)

        # Final dense layers
        last_point_filters = _make_divisible(last_point_filters * width_multiplier)
        x = layers.Dense(last_point_filters, use_bias=True, name="dense_1")(x)
        x = HardSwish(name="activation_dense")(x)

        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate, name="dropout")(x)
        
        x = layers.Dense(
            num_classes, use_bias=True, name="predictions"
        )(x)
    else:
        if pooling == "avg":
            x = layers.GlobalAveragePooling2D(name="avg_pool")(x)
        elif pooling == "max":
            x = layers.GlobalMaxPooling2D(name="max_pool")(x)

    # Create model
    model =tf.keras.Model(inputs=inputs, outputs=x, name=f"MobileNetV3_{architecture}")

    return model  


### Convenience Function

In [10]:
def MobileNetV3Large(
        input_shape=(224, 224, 3),
        num_classes=1000,
        width_multiplier=1.0,
        include_top=True,
        pooling=None,
        dropout_rate=0.2
):
    """
    Build MobileNetV3-Large model.
    """

    return MobileNetV3(
        architecture="large",
        input_shape=input_shape,
        num_classes=num_classes,
        width_multiplier=width_multiplier,
        include_top=include_top,
        pooling=pooling,
        dropout_rate=dropout_rate
    )

In [11]:
def MobileNetV3Small(
        input_shape=(224, 224, 3),
        num_classes=1000,
        width_multiplier=1.0,
        include_top=True,
        pooling=None,
        dropout_rate=0.2
):
    """
    Build MobileNetV3-Small model.
    """

    return MobileNetV3(
        architecture="small",
        input_shape=input_shape,
        num_classes=num_classes,
        width_multiplier=width_multiplier,
        include_top=include_top,
        pooling=pooling,
        dropout_rate=dropout_rate
    )

### Building MobileNetV3-Large...

In [ ]:
print("Building MobileNetV3-Large...")
model_large = MobileNetV3Large(
    input_shape=INPUT_SHAPE,
    num_classes=1,
    width_multiplier=1.0,
    include_top=True,
    dropout_rate=0.2
)

print(f"Model created: {model_large.name}")
print(f"Total parameters: {model_large.count_params():, }")
model_large.summary()

### Building MobileNetV3-Small ...

In [ ]:
print("Building MobileNetV3-Small...")
model_small = MobileNetV3Small(
    input_shape=INPUT_SHAPE,
    num_classes=1,
    width_multiplier=1.0,
    include_top=True,
    dropout_rate=0.2
)

print(f"Model created: {model_small.name}")
print(f"Total parameters: {model_small.count_params():, }")
model_small.summary()

In [ ]:
model = model_large
# model = model_small # Uncomment if training small and comment the above

In [ ]:
# Compile Model
loss_function = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer=tf.keras.optimizers.Adam(learning_rate=0.000005)
model.compile(
    loss=loss_function,
    optimizer=optimizer,
    metrics=["accuracy"]
)

In [ ]:
# Configure Callbacks

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath="models/vanilla_mobilenet_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    verbose=1,
    restore_best_weights=True
)

reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    patience=5,
    factor=0.3,
    verbose=1
)

callbacks = [model_checkpoint, early_stopping, reduce_learning_rate]


In [ ]:
# Train the Model to learn patterns from the image

history = model.fit(train_dataset,
                    validation_data=validation_dataset,
                    epochs=EPOCHS,
                    callbacks=callbacks)

In [ ]:
def plot_learning_curves(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(len(acc))


    plt.figure(figsize=(18, 7))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="Training Accuracy")
    plt.plot(epochs_range, val_acc, label="Validation Accuracy")
    plt.legend()
    plt.title("Accuracy")

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="Training Loss")
    plt.plot(epochs_range, val_loss, label="Validation Loss")
    plt.legend()
    plt.title("Loss")

    plt.show()


In [ ]:
loss, accuracy = model.evaluate(validation_dataset)

print(f"Model Loss: {loss:.2f}")
print(f"Model Accuracy: {accuracy:.2f}")